In [0]:
%sql
-- Create Date Dimension Table
CREATE OR REPLACE TABLE ecommerce.e_comm_gold.dimDate AS
SELECT 
    -- Date Key (numeric: YYYYMMDD format)
    CAST(date_format(date_col, 'yyyyMMdd') AS INT) AS date_key,
    
    -- Full Date
    date_col AS full_date,
    
    -- Year attributes
    year(date_col) AS year,
    
    -- Quarter attributes
    quarter(date_col) AS quarter,
    CONCAT('Q', quarter(date_col), '-', year(date_col)) AS quarter_name,
    
    -- Month attributes
    month(date_col) AS month_num,
    date_format(date_col, 'MMMM') AS month_name,
    date_format(date_col, 'MMM') AS month_short_name,
    CONCAT(year(date_col), '-', lpad(month(date_col), 2, '0')) AS year_month,
    
    -- Week attributes
    weekofyear(date_col) AS week_of_year,
    
    -- Day attributes
    dayofmonth(date_col) AS day_of_month,
    dayofyear(date_col) AS day_of_year,
    dayofweek(date_col) AS day_of_week_num,
    date_format(date_col, 'EEEE') AS day_of_week_name,
    date_format(date_col, 'E') AS day_of_week_short,
    
    -- Business day flag (Monday-Friday = 1, Weekend = 0)
    CASE WHEN dayofweek(date_col) IN (1, 7) THEN 0 ELSE 1 END AS is_weekday,
    
    -- Relative dates
    CASE WHEN date_col = current_date() THEN 1 ELSE 0 END AS is_today,
    CASE WHEN year(date_col) = year(current_date()) THEN 1 ELSE 0 END AS is_current_year,
    CASE WHEN year(date_col) = year(current_date()) AND month(date_col) = month(current_date()) THEN 1 ELSE 0 END AS is_current_month,
    
    -- Audit
    current_timestamp() AS created_date

FROM (
    -- Generate date range: 2020-01-01 to 2030-12-31
    SELECT explode(sequence(
        to_date('2020-01-01'),
        to_date('2030-12-31'),
        interval 1 day
    )) AS date_col
)
ORDER BY date_col;